In [8]:
import pandas as pd

df = pd.read_csv('train.csv')

df = df[['time', 'pv']]

In [9]:
df.head()

,time,pv
0,2024-02-28 21:00:00+00:00,0.0
1,2024-02-28 21:30:00+00:00,0.0
2,2024-02-28 22:00:00+00:00,NaN
3,2024-02-28 22:30:00+00:00,NaN
4,2024-02-28 23:00:00+00:00,NaN


In [ ]:
mask = (df['time'] >= '2024-06-01') & (df['time'] <= '2024-07-08')
df = df.loc[mask]
df.tail()

,time,pv
6241,2024-07-07 21:30:00+00:00,0.0
6242,2024-07-07 22:00:00+00:00,NaN
6243,2024-07-07 22:30:00+00:00,NaN
6244,2024-07-07 23:00:00+00:00,NaN
6245,2024-07-07 23:30:00+00:00,NaN


In [11]:
df['pv'] = df['pv'].fillna(0)

df = df.reset_index(drop=True)
df.tail()

,time,pv
1771,2024-07-07 21:30:00+00:00,0.0
1772,2024-07-07 22:00:00+00:00,0.0
1773,2024-07-07 22:30:00+00:00,0.0
1774,2024-07-07 23:00:00+00:00,0.0
1775,2024-07-07 23:30:00+00:00,0.0


In [ ]:
import numpy as np, pandas as pd

# ---------- 0. Time axis ----------
idx = pd.date_range("2025-06-01", "2025-07-07 23:30",
                    freq="30min")

# helper masks
is_weekend = idx.weekday >= 5          # Sat=5, Sun=6
half_hour  = idx.hour + idx.minute/60  # fractional hour in day
dt_h = 0.5                              # 30-min in hours

# ---------- 1. PV generation ----------


# ---------- 2. Uncontrolled base-load ----------
def weekday_profile(h):
    # lights on 07–08 h; evening bump 18–24 h
    if   0 <= h < 6:   return 450
    elif 6 <= h < 9:   return 600     # kettle, lights
    elif 9 <= h < 17:  return 550
    elif 17<= h < 24:  return 2000     # TV, lights
def weekend_profile(h):
    # user home—higher daytime loads, plus lunch cooking mid-day
    if   0 <= h < 6:   return 500
    elif 6 <= h < 12:  return 1000
    elif 12<= h < 14:  return 2000     # stove + lights for lunch
    elif 14<= h < 18:  return 800
    elif 18<= h < 24:  return 2500

base_w = np.empty(len(idx))
for i, ts in enumerate(idx):
    h = ts.hour + ts.minute/60
    base_w[i] = (weekend_profile(h) if is_weekend[i]
                  else weekday_profile(h))
base_w += np.random.normal(0, 100, len(idx))      # make it noisy

# ---------- 3. Device schedules (fixed routines) ----------

# 3.1  Air-conditioner (1.8 kW)
ac_w = np.zeros(len(idx))
# week-day pattern
mask = (~is_weekend &
       (((18 <= idx.hour) & (idx.hour < 20)) |
        ((idx.hour >= 22) | (idx.hour < 3))))
ac_w[mask] = 1800
# week-end pattern
mask = (is_weekend &
       (((12 <= idx.hour) & (idx.hour < 15)) |
        ((18 <= idx.hour) & (idx.hour < 20)) |
        ((idx.hour >= 23) | (idx.hour < 2))))
ac_w[mask] = 1800

# 3.2  Water-heater (2.4 kW, daily)
water_w = (((idx.hour==6)  & (idx.minute<90)) |
            ((idx.hour==18) & (idx.minute<90))).astype(float)*2400

# 3.3  Washer / Dryer (every 3 days)
washer_w = np.zeros(len(idx))
dryer_w  = np.zeros(len(idx))
day_number = (idx.normalize() - idx[0].normalize()).days
is_laundry_day = (day_number % 3 == 0)

# Washer 18:00–18:45
mask = (is_laundry_day &
        (idx.hour == 18) & (idx.minute < 45))
washer_w[mask] = 1000
# Dryer 18:45–19:30
mask = (is_laundry_day &
        ((idx.hour == 18) & (idx.minute >= 45)))
dryer_w[mask] = 1500

# ---------- 4. Assemble & export ----------
total_w = (base_w + ac_w + water_w +
            washer_w + dryer_w)


household_power = pd.DataFrame({
    "base_w":  base_w.clip(0),
    "ac_w":    ac_w,
    "water_w": water_w,
    "washer_w": washer_w,
    "dryer_w":  dryer_w,
    "total_w": total_w
}, index=idx)

In [ ]:
household_power.head()

,base_w,ac_w,water_w,washer_w,dryer_w,total_w
2025-07-07 21:30:00,1978.847794,0.0,0.0,0.0,0.0,1978.847794
2025-07-07 22:00:00,1925.389486,1800.0,0.0,0.0,0.0,3725.389486
2025-07-07 22:30:00,1903.555975,1800.0,0.0,0.0,0.0,3703.555975
2025-07-07 23:00:00,1913.738685,1800.0,0.0,0.0,0.0,3713.738685
2025-07-07 23:30:00,1986.491711,1800.0,0.0,0.0,0.0,3786.491711


In [14]:
household_power['pv'] = df['pv'].values
household_power = household_power.reset_index().rename(columns={'index': 'time'})
household_power.tail()

,time,base_w,ac_w,water_w,washer_w,dryer_w,total_w,pv
1771,2025-07-07 21:30:00,1978.847794,0.0,0.0,0.0,0.0,1978.847794,0.0
1772,2025-07-07 22:00:00,1925.389486,1800.0,0.0,0.0,0.0,3725.389486,0.0
1773,2025-07-07 22:30:00,1903.555975,1800.0,0.0,0.0,0.0,3703.555975,0.0
1774,2025-07-07 23:00:00,1913.738685,1800.0,0.0,0.0,0.0,3713.738685,0.0
1775,2025-07-07 23:30:00,1986.491711,1800.0,0.0,0.0,0.0,3786.491711,0.0


In [ ]:
# Select and rename columns
export_df = household_power[['time', 'total_w', 'pv']].rename(
    columns={
        'total_w': 'sensor.power_load_no_var_loads',
        'pv': 'sensor.power_photovoltaics'
    }
)

# Export to CSV
export_df.to_csv('sensor.power_total_load.csv', index=False)

In [ ]:
household_power = household_power.rename(
    columns={
        'total_w': 'sensor.power_load_no_var_loads',
        'pv': 'sensor.power_photovoltaics'
    }
)
household_power.tail()

,time,base_w,ac_w,water_w,washer_w,dryer_w,sensor.power_load_no_var_loads,sensor.power_photovoltaics
1771,2025-07-07 21:30:00,1978.847794,0.0,0.0,0.0,0.0,1978.847794,0.0
1772,2025-07-07 22:00:00,1925.389486,1800.0,0.0,0.0,0.0,3725.389486,0.0
1773,2025-07-07 22:30:00,1903.555975,1800.0,0.0,0.0,0.0,3703.555975,0.0
1774,2025-07-07 23:00:00,1913.738685,1800.0,0.0,0.0,0.0,3713.738685,0.0
1775,2025-07-07 23:30:00,1986.491711,1800.0,0.0,0.0,0.0,3786.491711,0.0


In [19]:
household_power.to_csv('household_power.csv', index=False)